In [17]:
# Assessment 4 - Part 1: Data Enrichment and Feature Engineering
# Research Questions:
#
# RQ1: Can we predict whether someone's highest non-school qualification
# is relevant to their current job, using demographic, education,
# employment, and occupation-income features?
#
# RQ2: Can migrant employee subgroups be classified as above or below
# median income using demographic, migration, and visa characteristics?
#
# Outputs:
# rq1_qualification_relevance_part1_dataset_v2.csv
# rq2_migrant_income_part1_dataset_v2.csv

In [18]:
# Import libraries 
from pathlib import Path
import re
import numpy as np
import pandas as pd

# Fix seed so results are reproducible
RANDOM_SAMPLE = 0
rng = np.random.default_rng(RANDOM_SAMPLE)

# File paths
QAW_FILE = Path(r"C:\Users\ASUS\Downloads\QAW2223DC (2).xlsx")
INCOME_OCCUPATION_FILE = Path(
    r"C:\Users\ASUS\Downloads\Table 4 - Employment income, earners and summary "
    r"statistics by industry and occupation of main job, 2021-22 to 2022-23.xlsx"
)
MIGRANT_INCOME_FILE = Path(
    r"C:\Users\ASUS\Downloads\Table 12 - Migrants, Employee income by arrival "
    r"group, 2018-19 to 2022-23.xlsx"
)
# Output directory for cleaned datasets
BASE_DIR = Path(r"C:\Users\ASUS\Downloads")
OUTPUT_DIR = BASE_DIR / "outputs_prt564_part1_v2"
DATA_DIR = OUTPUT_DIR / "cleaned_data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [19]:
# Helper functions
# Function 1: This function strips newlines, fixes common mojibake, standardises whitespace
# and returns NaN for empty or placeholder values.
def clean_text(value):
    if pd.isna(value):
        return np.nan
    value = str(value).replace("\n", " ")

    # To fix encoding issue occurred in the testing phase 
    replacements = {
        "â€“": "-",
        "â€”": "-",
        "â€˜": "'",
        "â€™": "'",
        "â€œ": '"',
        "â€�": '"',
        "–": "-",
        "—": "-"
    }
    for old, new in replacements.items():
        value = value.replace(old, new)

    value = re.sub(r"\s+", " ", value).strip()

    if value in ["", "-", "..", "nan", "NaN"]:
        return np.nan
    return value

In [20]:
# Function 2: To parse numeric cells that are set as strings with commas or dashes 
def clean_number(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, str):
        value = value.replace(",", "").strip()
        if value in ["", "-", "..", "na", "n/a", "np", "NaN"]:
            return np.nan
    try:
        return float(value)
    except ValueError:
        return np.nan

In [21]:
# Function 3: To standardise text: lowercase, replace & with 'and', remove symbols, 
# and collapse whitespace, use for creating merge keys
def same_text(value):
    value = clean_text(value)
    if pd.isna(value):
        return np.nan
    value = value.lower().replace("&", "and")
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()

In [22]:
# Function 4: To extract the midpoint from age group labels like '35 to 44 years' → 39.5
def age_midpoint(age_group):
    age_group = clean_text(age_group)
    if pd.isna(age_group):
        return np.nan
    nums = [float(n) for n in re.findall(r"\d+", age_group)]
    if len(nums) >= 2:
        return (nums[0] + nums[1]) / 2
    if len(nums) == 1:
        return nums[0]
    return np.nan

In [23]:
# Function 5: To turn aggregated data into sample records 
# For example, a row with estimate_000 = 5 will become 
# 5 identical rows with the same features except for the count
def expand_records(df, count_col="estimate_000", scale=1):
    rows = []
    for _, row in df.iterrows():
        count = clean_number(row[count_col])
        if pd.isna(count) or count <= 0:
            continue
        n = max(int(round(count * scale)), 1)
        row_dict = row.drop(labels=[count_col]).to_dict()
        rows.extend([row_dict] * n)
    return pd.DataFrame(rows)

In [24]:
# Function 6: To sample new values for each instance of a particular feature,
# taking into account the marginal distribution of the feature across each class of the target
def sample_feature(records, distribution_df, new_feature_name, target_col="target"):
    records = records.copy()
    for target_value in records[target_col].dropna().unique():
        mask = records[target_col] == target_value
        subset = distribution_df[distribution_df[target_col] == target_value].copy()
        subset = subset.dropna(subset=["category", "estimate_000"])

        if subset.empty:
            records.loc[mask, new_feature_name] = "Unknown"
            continue

        weights = subset["estimate_000"].astype(float).to_numpy()
        weights = weights / weights.sum()
        categories = subset["category"].astype(str).to_numpy()
        # Sample values for the new feature based on the distribution for this target class
        records.loc[mask, new_feature_name] = rng.choice(
            categories, size=mask.sum(), p=weights
        )
    return records


In [25]:
# RQ1: Qualification relevance
# Function 7: To read Table 4.2 and create occupation-level income features
def load_occupation_income_table4_2():
    raw = pd.read_excel(INCOME_OCCUPATION_FILE, sheet_name="Table 4.2", header=None)
    income = raw.iloc[7:16, [0, 2, 4, 6, 8, 10]].copy()
    income.columns = [
        "occupation",
        "occupation_earners_2022_23",
        "occupation_median_age_2022_23",
        "occupation_sum_income_2022_23",
        "occupation_median_income_2022_23",
        "occupation_mean_income_2022_23",
    ]
    # Clean the occupation names and numeric columns
    income["occupation"] = income["occupation"].apply(clean_text)
    for col in income.columns[1:]:
        income[col] = income[col].apply(clean_number)

    # Use the Australia-wide row as the benchmark; fall back to column median if missing
    nat = income[income["occupation"].str.lower() == "australia"]
    national_median = (
        nat["occupation_median_income_2022_23"].iloc[0]
        if not nat.empty
        else income["occupation_median_income_2022_23"].median()
    )

    income = income[income["occupation"].str.lower() != "australia"].copy()
    income["occupation_key"] = income["occupation"].apply(same_text)
    income["occupation_income_level"] = np.where(
        income["occupation_median_income_2022_23"] >= national_median,
        "Above national occupation median",
        "Below national occupation median",
    )
    income["occupation_income_rank"] = (
        income["occupation_median_income_2022_23"]
        .rank(ascending=False, method="dense")
        .astype(int)
    )
    return income

In [26]:
# Function 8: To read Table 10 and create the base dataset of records with employment status 
def load_q1_base_from_qaw_table10():
    raw = pd.read_excel(QAW_FILE, sheet_name="Table 10", header=None)
    section = raw.iloc[8:85, :6].copy()

    rows = []
    current_employment_status = None
    current_sex = None

    for _, row in section.iterrows():
        label = clean_text(row.iloc[0])
        if pd.isna(label):
            continue

        if label == "Employed full-time":
            current_employment_status = "Employed full-time"
            current_sex = None
        elif label == "Employed part-time":
            current_employment_status = "Employed part-time"
            current_sex = None
        elif label == "Total employed":
            current_employment_status = None
            current_sex = None
        elif label in ["Males", "Females"]:
            current_sex = label
        elif label == "Persons":
            current_sex = None  # skip persons aggregates
        elif "Total" in label:
            continue
        elif (
            "years" in label
            and current_employment_status in ["Employed full-time", "Employed part-time"]
            and current_sex in ["Males", "Females"]
        ):
            rows.append({
                "employment_status": current_employment_status,
                "sex": current_sex,
                "age_group": label,
                "target": "Relevant",
                "estimate_000": clean_number(row.iloc[3]),
            })
            rows.append({
                "employment_status": current_employment_status,
                "sex": current_sex,
                "age_group": label,
                "target": "Not relevant",
                "estimate_000": clean_number(row.iloc[4]),
            })

    base = pd.DataFrame(rows)
    records = expand_records(base, count_col="estimate_000", scale=1)
    records["age_midpoint"] = records["age_group"].apply(age_midpoint)
    return records


In [27]:
# Function 9: To read Table 6 and create marginal distributions for sampling additional features
def load_qaw_table6_distribution(start_row, end_row):
    raw = pd.read_excel(QAW_FILE, sheet_name="Table 6", header=None)
    part = raw.iloc[start_row:end_row, [0, 3, 4]].copy()
    part.columns = ["category", "Relevant", "Not relevant"]
    part["category"] = part["category"].apply(clean_text)

    rows = []
    for _, row in part.iterrows():
        for target in ["Relevant", "Not relevant"]:
            val = clean_number(row[target])
            if not pd.isna(val) and val > 0:
                rows.append({"target": target, "category": row["category"], "estimate_000": val})
    return pd.DataFrame(rows)

In [28]:
# Function 10: To read Table 11 and create occupation distributions for sampling occupation feature
def load_qaw_table11_occupation_distribution():
    raw = pd.read_excel(QAW_FILE, sheet_name="Table 11", header=None)
    occupations = [clean_text(x) for x in raw.iloc[5, 1:9].tolist()]
    relevant_counts = [clean_number(x) for x in raw.iloc[8, 1:9].tolist()]
    not_relevant_counts = [clean_number(x) for x in raw.iloc[11, 1:9].tolist()]

    rows = []
    for occ, rel, not_rel in zip(occupations, relevant_counts, not_relevant_counts):
        rows.append({"target": "Relevant", "category": occ, "estimate_000": rel})
        rows.append({"target": "Not relevant", "category": occ, "estimate_000": not_rel})
    return pd.DataFrame(rows)

In [29]:
# Function 11: To assemble the RQ1 dataset by sampling features based on Table 6 
# distributions and merging occupation income features
def build_rq1_dataset():
    q1 = load_q1_base_from_qaw_table10()
    # Features we can sample from Table 6 marginal distributions
    feature_distributions = {
        "number_of_non_school_qualifications": load_qaw_table6_distribution(19, 22),
        "qualification_recency": load_qaw_table6_distribution(23, 25),
        "current_job_skill_level": load_qaw_table6_distribution(26, 31),
        "country_of_birth_group": load_qaw_table6_distribution(55, 57),
        "citizenship_status": load_qaw_table6_distribution(65, 67),
    }
    for feature_name, dist in feature_distributions.items():
        q1 = sample_feature(q1, dist, feature_name)

    # Sample occupation from Table 11, then merge income features
    q1 = sample_feature(q1, load_qaw_table11_occupation_distribution(), "occupation")
    q1["occupation_key"] = q1["occupation"].apply(same_text)

    occupation_income = load_occupation_income_table4_2()
    q1 = q1.merge(occupation_income, on="occupation_key", how="left",suffixes=("", "_from_income_table"))
    q1 = q1.drop(columns=["occupation_key", "occupation_from_income_table"], errors="ignore")

    rq1_path = DATA_DIR / "rq1_qualification_relevance_part1_dataset_v2.csv"
    q1.to_csv(rq1_path, index=False, encoding="utf-8-sig")
    print("RQ1 dataset saved to:", rq1_path)
    print("RQ1 shape:", q1.shape)
    return q1


In [30]:
# RQ2: Migrant income group
# Function 12: To read the three Table 12.x sheets, filter to relevant rows, and create the RQ2 dataset
def load_migrant_income_sheet(sheet_name, arrival_group):
    raw = pd.read_excel(MIGRANT_INCOME_FILE, sheet_name=sheet_name, header=None)
    df = raw.iloc[7:, [1, 2, 3, 4, 5, 10, 20]].copy()
    df.columns = [
        "state", "visa_group", "age_range", "applicant_status", "sex",
        "earners_2022_23", "median_employee_income_2022_23",
    ]
    df = df.dropna(how="all")
    # Clean text columns and numeric columns
    for col in ["state", "visa_group", "age_range", "applicant_status", "sex"]:
        df[col] = df[col].apply(clean_text)
    for col in ["earners_2022_23", "median_employee_income_2022_23"]:
        df[col] = df[col].apply(clean_number)

    df["arrival_group"] = arrival_group
    df["age_midpoint"] = df["age_range"].apply(age_midpoint)
    return df


In [31]:
def build_rq2_dataset():
    q2 = pd.concat([
        load_migrant_income_sheet("Table 12.1", "Less than 5 years since arrival"),
        load_migrant_income_sheet("Table 12.2", "5 to 10 years since arrival"),
        load_migrant_income_sheet("Table 12.3", "More than 10 years since arrival"),
    ], ignore_index=True)

    # Lowercase columns for filtering
    q2["state_clean"]     = q2["state"].astype(str).str.lower().str.strip()
    q2["sex_clean"]       = q2["sex"].astype(str).str.lower().str.strip()
    q2["age_clean"]       = q2["age_range"].astype(str).str.lower().str.strip()
    q2["applicant_clean"] = q2["applicant_status"].astype(str).str.lower().str.strip()

    age_groups_sum = ["24 and under", "25 to 34", "35 to 44","45 to 54", "55 to 64", "65+"]

    q2 = q2[
        (q2["state_clean"] == "australia") &
        (q2["sex_clean"].isin(["males", "females"])) &
        (q2["age_clean"].isin(age_groups_sum)) &
        (q2["applicant_clean"].isin(["primary", "secondary"])) &
        (q2["median_employee_income_2022_23"].notna()) &
        (q2["earners_2022_23"].notna()) &
        (q2["earners_2022_23"] > 0)
    ].copy()

    if q2.empty:
        raise ValueError("RQ2 dataset is empty - check Table 12 filters and sheet layout.")
    # Create binary target based on whether median income is above or below the overall median across all records
    benchmark_median = q2["median_employee_income_2022_23"].median()
    q2["income_target"] = np.where(
        q2["median_employee_income_2022_23"] >= benchmark_median,
        "Above median income",
        "Below median income",
    )
    # Drop columns that were only needed to derive the target 
    q2 = q2.drop(columns=[
        "state", "state_clean", "sex_clean", "age_clean", "applicant_clean",
        "earners_2022_23", "median_employee_income_2022_23",
    ], errors="ignore")

    rq2_path = DATA_DIR / "rq2_migrant_income_part1_dataset_v2.csv"
    q2.to_csv(rq2_path, index=False, encoding="utf-8-sig")
    print("RQ2 dataset saved to:", rq2_path)
    print("RQ2 shape:", q2.shape)
    return q2

In [32]:
# Main execution
def main():
    build_rq1_dataset()
    build_rq2_dataset()

if __name__ == "__main__":
    main()

RQ1 dataset saved to: C:\Users\ASUS\Downloads\outputs_prt564_part1_v2\cleaned_data\rq1_qualification_relevance_part1_dataset_v2.csv
RQ1 shape: (9946, 18)
RQ2 dataset saved to: C:\Users\ASUS\Downloads\outputs_prt564_part1_v2\cleaned_data\rq2_migrant_income_part1_dataset_v2.csv
RQ2 shape: (284, 7)
